# PPO Training on Colab

Use this notebook only after a TinyLlama LoRA/merged model run exists in Google Drive. PPO uses TRL value-head models and requires CUDA; local CPU or disk offload is expected to fail.

In [ ]:
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from datetime import datetime, timezone

REPO_URL = 'https://github.com/ashioyajotham/weather_forecasting_lora.git'
os.environ['REPO_URL'] = REPO_URL

# Set this to the TinyLlama/GGUF notebook run directory in Drive.
SOURCE_RUN_DIR = '/content/drive/MyDrive/weather_forecasting_lora_runs/tinyllama_gguf_REPLACE_ME'

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
DRIVE_ROOT = '/content/drive/MyDrive/weather_forecasting_lora_runs'
RUN_DIR = f'{DRIVE_ROOT}/ppo_{RUN_ID}'
os.environ['RUN_DIR'] = RUN_DIR
os.makedirs(RUN_DIR, exist_ok=True)
print('SOURCE_RUN_DIR=', SOURCE_RUN_DIR)
print('RUN_DIR=', RUN_DIR)

In [ ]:
!rm -rf /content/weather_forecasting_lora
!git clone {REPO_URL} /content/weather_forecasting_lora
%cd /content/weather_forecasting_lora
!git rev-parse HEAD

In [ ]:
!python -m pip install -U pip
!python -m pip install -r requirements-colab.txt
!python -m pip install bitsandbytes

In [ ]:
import os
import shutil
from pathlib import Path

os.environ['USE_TF'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['WANDB_DISABLED'] = 'true'
os.environ['WEATHER_PPO_MAX_SAMPLES'] = os.environ.get('WEATHER_PPO_MAX_SAMPLES', '64')

source = Path(SOURCE_RUN_DIR)
if 'REPLACE_ME' in SOURCE_RUN_DIR or not source.exists():
    raise RuntimeError('Set SOURCE_RUN_DIR to a real TinyLlama/GGUF Drive run directory')

merged_source = source / 'merged'
if not merged_source.exists():
    raise RuntimeError(f'Missing merged artifact: {merged_source}')

target = Path('models/weather-merged')
target.parent.mkdir(parents=True, exist_ok=True)
if target.exists():
    shutil.rmtree(target)
shutil.copytree(merged_source, target)
print('Imported merged model to', target)

In [ ]:
!python scripts/smoke_check.py
!python scripts/ppo_smoke.py
!python train_ppo.py 2>&1 | tee "$RUN_DIR/ppo_dry_run.log"

## Small PPO trial

Start with 64 samples on T4. Increase only after confirming memory headroom and stable rewards.

In [ ]:
!python train_ppo.py --run-training --limit "$WEATHER_PPO_MAX_SAMPLES" --output-dir models/weather-lora-ppo 2>&1 | tee "$RUN_DIR/ppo_train.log"

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

ppo_path = Path('models/weather-lora-ppo')
ppo_target = Path(RUN_DIR) / 'ppo_model'
if ppo_path.exists():
    if ppo_target.exists():
        shutil.rmtree(ppo_target)
    shutil.copytree(ppo_path, ppo_target)
else:
    print('missing PPO output', ppo_path)

def path_size(path: Path) -> int:
    if not path.exists():
        return 0
    if path.is_file():
        return path.stat().st_size
    return sum(p.stat().st_size for p in path.rglob('*') if p.is_file())

manifest = {
    'run_id': RUN_ID,
    'source_run_dir': SOURCE_RUN_DIR,
    'repo_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip(),
    'gpu': subprocess.getoutput("nvidia-smi --query-gpu=name --format=csv,noheader | head -n 1"),
    'weather_ppo_max_samples': os.environ.get('WEATHER_PPO_MAX_SAMPLES'),
    'ppo_model': {'path': str(ppo_target), 'bytes': path_size(ppo_target)},
}
manifest_path = Path(RUN_DIR) / 'manifest_ppo.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps(manifest, indent=2))

## Acceptance

Treat PPO as experimental until the exported PPO model is imported locally and compared against the pinned generation-quality eval. Do not replace the SFT/GGUF artifact unless PPO improves reward behavior without degrading forecast quality.